# C05：Algebra/HCL一阶与多阶候选筛选对照
只新增10次一阶训练，已有10次多阶结果从baseline_reference读取。
输入始终[phi,xi10]、只切换筛选score，600/50/1000/5预算不变；不重算缓存。
命令行和Notebook二选一启动。本机已后台运行时不再执行本Notebook的正式运行单元。

In [ ]:
from pathlib import Path
import os,sys,subprocess,signal
import torch  # 请选择已安装PyTorch的内核；服务器使用之前通过测试的CNN
ROOT=Path.cwd().resolve()
assert (ROOT/'plan_order_ablation.json').is_file(), '请在新对照目录打开Notebook'
PLAN=ROOT/'plan_order_ablation.json'
# 本机默认路径；服务器上传后改成DATA_ROOT=ROOT.parent，CACHE_ROOT=ROOT.parent/'Hyper_Distance'
DATA_ROOT=Path('D:/YX/HyperGraphMatrix')
CACHE_ROOT=Path('D:/YX/Hyper_Distance')
OUTPUT=ROOT/'results/first_order'
WORKERS=3
# Windows加载检查点I/O修复，子进程继承；Linux不需要。
if os.name=='nt':
    os.environ['PYTHONPATH']=str(ROOT/'.ops')+os.pathsep+os.environ.get('PYTHONPATH','')
def run_cli(*args):
    cmd=[sys.executable,'-u',str(ROOT/'run_suite.py'),*map(str,args)]
    process=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,start_new_session=(os.name=='posix'))
    try:
        for line in process.stdout:print(line,end='',flush=True)
        status=process.wait()
        if status:raise subprocess.CalledProcessError(status,cmd)
    except BaseException:
        if process.poll() is None:
            if os.name=='posix':os.killpg(process.pid,signal.SIGINT)
            else:process.terminate()
        raise
print(sys.executable)

## 工程检查与输入检查

In [ ]:
subprocess.run([sys.executable,'-m','unittest','test_order_ablation','-v'],cwd=ROOT,check=True)
run_cli('check','--plan',PLAN,'--data-root',DATA_ROOT,'--cache-root',CACHE_ROOT)

## 正式运行新增10项（已有结果和检查点自动接续）

In [ ]:
run_cli('run','--plan',PLAN,'--data-root',DATA_ROOT,'--cache-root',CACHE_ROOT,
        '--output',OUTPUT,'--workers',WORKERS,'--s2v-workers',1)

## 全部结束后自动校验配对数据并汇总

In [ ]:
subprocess.run([sys.executable,str(ROOT/'summarize_order_ablation.py')],cwd=ROOT,check=True)
print((ROOT/'comparison/结果分析.md').read_text(encoding='utf-8'))